# Laboratorio 1 — Exploración, preparación y regresión lineal
## Caso AlpesPlanck: predicción de la temperatura máxima del día siguiente

**Curso:** ISIS-2611 Aprendizaje Automático

**Integrantes:** `Nikol Katherin Rodriguez Ortiz` — `<Nombre Estudiante 2>`

---

### Contexto

AlpesPlanck registra variables meteorológicas diarias (presión, humedad, viento y calendario) y quiere
**predecir la temperatura máxima del día siguiente** e **identificar qué variables meteorológicas
aportan más a esa predicción**, con el fin de anticipar riesgos como incendios forestales o los efectos
de fenómenos climáticos como "El Niño"/"La Niña".

### Objetivos

1. Construir un modelo de regresión que estime la temperatura máxima del día siguiente siguiendo el
   ciclo de machine learning (entender los datos → prepararlos → modelar → evaluar → comunicar).
2. Determinar las principales variables meteorológicas que permiten esa predicción.
3. Aplicar y comprender un modelo de regresión lineal, verificando formalmente sus supuestos.
4. Reconocer posibles fuentes de sesgo del modelo.
5. Comunicar los resultados de forma clara y sintética.

### Restricción metodológica del enunciado

> División entrenamiento–prueba con `random_state=42` y `test_size=0.25`, fijada para todo el notebook.

### Cómo usar este notebook

Este notebook trae **toda la arquitectura técnica lista y probada** (pipelines, modelos, validación
cruzada, supuestos, importancia de variables). Lo que falta —y es la parte que más vale en la
rúbrica— son **tus propios hallazgos**: cada sección de exploración trae el código de diagnóstico
para que lo ejecutes con tus datos reales, y un bloque `> ✍️ Tu hallazgo:` para que documentes lo que
observas y por qué tomas cada decisión de limpieza. No copies un hallazgo sin haberlo verificado en
tu propia salida: los números cambian si tu archivo de datos no es idéntico al de este ejemplo.

### Contenido

1. [Carga de datos y diccionario](#s1)
2. [Exploración de los datos](#s2)
3. [Preparación de los datos: pipeline de limpieza y transformación](#s3)
4. [Construcción de los modelos](#s4)
5. [Evaluación cuantitativa: tabla comparativa](#s5)
6. [Verificación de los supuestos de la regresión lineal](#s6)
7. [Importancia de variables](#s7)
8. [Selección del modelo final y predicciones sobre el conjunto de entrega](#s8)
9. [Análisis de resultados](#s9)
10. [Uso de herramientas de IA generativa](#s10)
11. [Guion para el video explicativo](#s11)

In [ ]:
# ============================================================
# Configuración del entorno
# ============================================================
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                      'figure.autolayout': True})
pd.set_option('display.width', 160, 'display.max_columns', 60)

# Semilla y partición fijadas por el enunciado: NO cambiar en ningún punto del notebook.
RANDOM_STATE = 42
TEST_SIZE = 0.25
np.random.seed(RANDOM_STATE)

print('numpy', np.__version__, '| pandas', pd.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)

<a name="s1"></a>
## 1. Carga de datos y diccionario

Trabajamos con tres archivos (ajusta los nombres si en tu entorno son distintos):

* `Datos Lab 1.csv` — datos **etiquetados** (contienen la variable objetivo `temp_max_manana`). Es el
  insumo para entrenar y evaluar. Lo llamamos *conjunto etiquetado*.
* `Datos Test Lab 1.csv` — datos **no etiquetados**, sobre los que hay que entregar la predicción
  final. Lo llamamos *conjunto de entrega*, para no confundirlo con la partición de prueba interna (el
  25 % que separamos del conjunto etiquetado).
* `Diccionario de datos.xlsx` — define unidades y rangos válidos de cada variable. **Léelo antes de
  explorar**: es la referencia contra la que vas a contrastar lo que encuentres.

In [ ]:
def resolve_data_dir():
    """Busca la carpeta que contiene los 3 archivos de datos, probando rutas típicas.
    Ajusta `candidates` si tu estructura de carpetas es distinta."""
    candidates = [Path.cwd(), Path.cwd() / 'data', Path.cwd().parent, Path.cwd().parent / 'data']
    required = ['Datos Lab 1.csv', 'Datos Test Lab 1.csv', 'Diccionario de datos.xlsx']
    for candidate in candidates:
        if all((candidate / f).exists() for f in required):
            return candidate
    raise FileNotFoundError(
        'No se encontró la carpeta de datos. Copia los 3 archivos junto al notebook o ajusta '
        'la lista `candidates` de resolve_data_dir().'
    )

DATA = resolve_data_dir()
etiquetado_raw = pd.read_csv(DATA / 'Datos Lab 1.csv')
entrega_raw    = pd.read_csv(DATA / 'Datos Test Lab 1.csv')
diccionario    = pd.read_excel(DATA / 'Diccionario de datos.xlsx')

TARGET  = 'temp_max_manana'   # ajusta si tu variable objetivo se llama distinto
ID_COLS = ['fecha']           # columnas identificadoras que NO son predictoras

print(f'Carpeta de datos     : {DATA}')
print(f'Conjunto etiquetado  : {etiquetado_raw.shape[0]:,} filas x {etiquetado_raw.shape[1]} columnas')
print(f'Conjunto de entrega  : {entrega_raw.shape[0]:,} filas x {entrega_raw.shape[1]} columnas')
print(f'Columna solo en el etiquetado: {set(etiquetado_raw.columns) - set(entrega_raw.columns)}')

In [ ]:
# El diccionario es el primer paso obligatorio: define unidades y rangos válidos,
# que son la base para detectar los errores de calidad en la sección 2.
diccionario

In [ ]:
etiquetado_raw.head()

> ✍️ **Primeras observaciones.** Mira con calma estas primeras filas contra el diccionario de datos:
> ¿hay algún valor que ya de entrada no calce con la unidad o el rango esperado (por ejemplo, un mes
> que no corresponde a la fecha, o una humedad que no parece estar en la escala que dice el
> diccionario)? Anota aquí lo que veas; lo cuantificamos en la sección 2.

<a name="s2"></a>
## 2. Exploración de los datos

Objetivo de esta sección: caracterizar la estructura del conjunto, **cuantificar** los problemas de
calidad de datos (no basta con intuirlos) y decidir, con justificación, cómo se van a tratar en la
sección 3. Una buena práctica es organizar los hallazgos bajo las cuatro dimensiones estándar de
calidad de datos:

| Dimensión | Pregunta que responde |
|:---|:---|
| **Completitud** | ¿Faltan valores que deberían estar? |
| **Unicidad** | ¿Hay registros repetidos que no deberían estarlo? |
| **Validez** | ¿Los valores caen dentro del dominio físico o de formato definido? |
| **Consistencia** | ¿Se contradice un dato con otro dato o con otra fuente del mismo hecho? |

Cada bloque de código produce evidencia; el bloque de texto que sigue es donde documentas el hallazgo
y la decisión de limpieza asociada — esa justificación es la que se evalúa.

### 2.1 Estructura y tipos de dato

In [ ]:
info = pd.DataFrame({
    'tipo'    : etiquetado_raw.dtypes.astype(str),
    'no_nulos': etiquetado_raw.notna().sum(),
    'nulos'   : etiquetado_raw.isna().sum(),
    '%_nulos' : (etiquetado_raw.isna().mean() * 100).round(2),
    'unicos'  : etiquetado_raw.nunique(),
})
info

> ✍️ **Tu hallazgo (completitud):** ¿todas las columnas tienen nulos o solo algunas? ¿el patrón
> parece disperso (típico de nulos inyectados al azar) o concentrado en bloques de filas (típico de
> falla de sensor)? ¿qué haces con las filas donde falta la variable objetivo, y por qué no es lo mismo
> tratarlas que tratar un nulo en un predictor?

### 2.2 Registros duplicados

In [ ]:
print(f'Filas exactamente duplicadas : {etiquetado_raw.duplicated().sum()}')

if 'fecha' in etiquetado_raw.columns:
    con_fecha = etiquetado_raw[etiquetado_raw['fecha'].notna()]
    dup_fecha = con_fecha[con_fecha['fecha'].duplicated(keep=False)]
    print(f'Fechas repetidas (valores únicos)      : {dup_fecha["fecha"].nunique()}')
    print(f'Filas involucradas en fechas repetidas : {len(dup_fecha)}')
    if len(dup_fecha):
        display(dup_fecha.sort_values('fecha').head(8))

> ✍️ **Tu hallazgo (unicidad):** ¿los duplicados son copias exactas, o hay filas con la misma fecha
> pero valores distintos ("duplicados sucios")? ¿los vas a eliminar, promediar o consolidar de otra
> forma? Justifica la decisión — recuerda que en el **conjunto de entrega no se debe eliminar ninguna
> fila**, porque hay que entregar una predicción para cada una.

### 2.3 Estadísticos descriptivos y comparación contra el conjunto de entrega

El conjunto de entrega es útil como referencia del rango físico esperado de cada variable, porque
(a diferencia del etiquetado) no fue alterado deliberadamente.

In [ ]:
num_cols_raw = etiquetado_raw.select_dtypes(include='number').columns
comparacion = pd.DataFrame({
    'min_etiq': etiquetado_raw[num_cols_raw].min(),
    'med_etiq': etiquetado_raw[num_cols_raw].median(),
    'max_etiq': etiquetado_raw[num_cols_raw].max(),
}).join(pd.DataFrame({
    'min_entrega': entrega_raw.min(numeric_only=True),
    'med_entrega': entrega_raw.median(numeric_only=True),
    'max_entrega': entrega_raw.max(numeric_only=True),
})).round(2)
comparacion

> ✍️ **Tu hallazgo (validez):** ¿qué variables tienen mínimos o máximos físicamente imposibles
> (por ejemplo, muy por fuera del rango del conjunto de entrega o del diccionario)? ¿reconoces algún
> valor "centinela" típico de datos faltantes codificados como número (p. ej. `-9999`)? Lístalos con su
> variable y decide: ¿corrección de escala, recorte, o tratarlo como nulo para que lo impute el
> pipeline?

### 2.4 Variables categóricas: consistencia de texto

In [ ]:
columnas_categoricas_raw = etiquetado_raw.select_dtypes(include='object').columns.tolist()
for c in columnas_categoricas_raw:
    u_e = sorted(etiquetado_raw[c].dropna().astype(str).unique())
    u_t = sorted(entrega_raw[c].dropna().astype(str).unique()) if c in entrega_raw.columns else []
    print(f'--- {c} ---')
    print(f'  etiquetado: {len(u_e):>3} categorías')
    print(f'  entrega   : {len(u_t):>3} categorías')
    if len(u_e) > len(u_t) * 1.5 or len(u_e) > 15:
        print(f'  primeras 15 categorías del etiquetado -> {u_e[:15]}')
    print()

> ✍️ **Tu hallazgo (consistencia):** ¿alguna columna categórica tiene muchas más categorías en el
> etiquetado que en la entrega? Eso suele indicar mezcla de mayúsculas/minúsculas, idiomas, errores de
> tecleo o espacios sobrantes. ¿Vas a normalizar el texto (`strip().lower()`), derivar la categoría de
> otra columna que sea confiable (por ejemplo, obtener el mes a partir de la fecha), o algo distinto?
> Justifica cuál te parece más robusto frente a categorías no vistas en el conjunto de entrega.

### 2.5 Distribución de variables y valores atípicos

In [ ]:
num_cols_raw = etiquetado_raw.select_dtypes(include='number').columns.drop(TARGET, errors='ignore')
n_cols_plot = min(len(num_cols_raw), 12)
fig, axes = plt.subplots(3, 4, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols_raw[:n_cols_plot]):
    etiquetado_raw[col].plot.hist(bins=40, ax=ax, color='#4c72b0')
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
etiquetado_raw[num_cols_raw[:n_cols_plot]].apply(
    lambda s: (s - s.mean()) / s.std()
).boxplot(ax=ax, rot=60)
ax.set_title('Boxplot de variables numéricas (estandarizadas) — outliers visibles como puntos alejados')
plt.show()

> ✍️ **Tu hallazgo:** ¿alguna variable muestra una distribución claramente bimodal, o con una
> "isla" de valores muy separada del resto (indicio de mezcla de escalas, como fracción 0–1 frente a
> porcentaje 0–100)? ¿qué outliers detectas y cómo decides tratarlos (recorte a un rango físico
> razonable, conversión a nulo para imputar, o dejarlos porque son válidos)?

### 2.6 Correlación con la variable objetivo

In [ ]:
correlaciones = (
    etiquetado_raw.select_dtypes(include='number')
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET, errors='ignore')
    .sort_values(key=abs, ascending=False)
)
fig, ax = plt.subplots(figsize=(7, 8))
correlaciones.plot.barh(ax=ax, color=np.where(correlaciones > 0, '#c44e52', '#4c72b0'))
ax.invert_yaxis()
ax.set_title(f'Correlación de Pearson con {TARGET}')
ax.axvline(0, color='black', lw=0.8)
plt.show()
correlaciones

> ✍️ **Tu hallazgo:** ¿qué variables numéricas correlacionan más fuerte (en valor absoluto) con la
> temperatura máxima de mañana? ¿el signo tiene sentido físico? Ten en cuenta que estas correlaciones
> están calculadas **sobre los datos crudos** (con los problemas de calidad todavía presentes), así que
> pueden cambiar bastante después de la limpieza — vale la pena repetir este cálculo más adelante sobre
> los datos ya limpios.

### 2.7 Resumen de hallazgos

Completa esta tabla con tus propios hallazgos (agrega o quita filas según lo que hayas encontrado) —
es el insumo directo para las reglas de limpieza de la sección 3.

| # | Hallazgo | Dimensión de calidad | Decisión de limpieza |
|:---:|:---|:---|:---|
| H1 | _completa_ | Completitud | _completa_ |
| H2 | _completa_ | Unicidad | _completa_ |
| H3 | _completa_ | Validez | _completa_ |
| H4 | _completa_ | Consistencia | _completa_ |
| … | | | |

<a name="s3"></a>
## 3. Preparación de los datos: pipeline de limpieza y transformación

Siguiendo el enunciado, **toda** la limpieza (corrección de escalas/unidades, tratamiento de
centinelas, imputación de nulos y estandarización) queda encapsulada en un `Pipeline` +
`ColumnTransformer` de scikit-learn. Esto tiene dos ventajas:

1. **Se puede reutilizar sin cambios** sobre el conjunto de entrega (que no debe perder ninguna fila).
2. **No hay fuga de información**: todo lo que se "aprende" de los datos (medianas para imputar, media
   y desviación para escalar) se ajusta únicamente con el conjunto de entrenamiento, nunca con el de
   prueba ni con el de entrega.

La única limpieza que queda **fuera** del pipeline es la que opera sobre filas (eliminar duplicados,
eliminar filas sin variable objetivo): esa limpieza solo tiene sentido en el conjunto etiquetado y
nunca se debe aplicar al conjunto de entrega.

### 3.1 Limpieza estructural (a nivel de fila) — solo conjunto etiquetado

In [ ]:
def limpieza_estructural(df, es_entrenamiento=True):
    """Limpieza que cambia el número de filas. Solo se aplica al conjunto etiquetado:
    el conjunto de entrega debe conservar TODAS sus filas para poder calificar cada predicción."""
    df = df.copy()
    if es_entrenamiento:
        antes = len(df)
        df = df.drop_duplicates()
        print(f'Duplicados exactos eliminados       : {antes - len(df)}')

        # TODO: si en la sección 2.2 detectaste duplicados "sucios" (misma fecha, valores distintos),
        # decide aquí si los consolidas (p. ej. promediando) en vez de solo usar drop_duplicates().

        antes = len(df)
        df = df.dropna(subset=[TARGET])
        print(f'Filas sin variable objetivo eliminadas: {antes - len(df)}')
        df = df.reset_index(drop=True)
    else:
        print('Conjunto de entrega: no se elimina ninguna fila (se debe predecir para todas).')
    return df

etiquetado = limpieza_estructural(etiquetado_raw, es_entrenamiento=True)
print(f'\nFilas finales del conjunto etiquetado: {len(etiquetado):,}')

### 3.2 Corrección de dominio (unidades, escalas, centinelas)

Esta es la transformación clave de la sección: corrige problemas de **validez** detectados en 2.3–2.5
(mezcla de escalas, errores de unidad, valores centinela). Es determinista —no se ajusta con
estadísticos del conjunto de entrenamiento—, así que puede vivir dentro del `Pipeline` sin riesgo de
fuga de información, y se aplica igual a entrenamiento, prueba y entrega.

In [ ]:
class CorrectorDominio(BaseEstimator, TransformerMixin):
    """Aplica correcciones deterministas de dominio (unidades, escalas, centinelas) ANTES de
    imputar y escalar. No aprende nada de los datos: las reglas se definen a partir del diccionario
    de datos y de tu propia exploración (sección 2), no de estadísticos del conjunto de entrenamiento.

    Parameters
    ----------
    reglas : dict[str, callable]
        {nombre_columna: funcion(serie) -> serie_corregida}
    """
    def __init__(self, reglas=None):
        self.reglas = reglas or {}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for columna, funcion in self.reglas.items():
            if columna in X.columns:
                X[columna] = funcion(X[columna])
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)


# --- Funciones de corrección disponibles (ejemplos genéricos; ajusta parámetros a tu hallazgo) ---
def corregir_fraccion_a_porcentaje(serie, umbral=1.0):
    """Si el valor está en escala 0-1, lo reescala a 0-100. Útil cuando detectas una variable
    de porcentaje con dos escalas mezcladas (hallazgo típico de humedad)."""
    serie = pd.to_numeric(serie, errors='coerce')
    return np.where(serie <= umbral, serie * 100, serie)

def corregir_factor_escala(serie, umbral, factor):
    """Si el valor supera `umbral`, se asume un error de escala (p. ej. x10) y se divide por `factor`."""
    serie = pd.to_numeric(serie, errors='coerce')
    return np.where(serie > umbral, serie / factor, serie)

def limpiar_centinelas(serie, valores_centinela=(-9999,)):
    """Convierte valores centinela conocidos (p. ej. -9999) en NaN para que el imputer los trate."""
    serie = pd.to_numeric(serie, errors='coerce')
    return serie.replace(list(valores_centinela), np.nan)


# --- Reglas de dominio: complétalas con lo que documentaste en la sección 2 ---
# Ejemplos (descomenta y ajusta el nombre de columna y los parámetros según tu propio hallazgo):
REGLAS_DOMINIO = {
    # 'humedad_media': corregir_fraccion_a_porcentaje,
    # 'humedad_min':   corregir_fraccion_a_porcentaje,
    # 'presion_media': lambda s: corregir_factor_escala(s, umbral=1100, factor=10),
    # 'rafaga_min':    limpiar_centinelas,
}
print(f'Reglas de dominio configuradas: {list(REGLAS_DOMINIO.keys())}')

### 3.3 Normalización de texto en categóricas (opcional)

Si en 2.4 detectaste una columna categórica con muchas variantes de escritura de la misma categoría,
puedes normalizar el texto aquí. Si prefieres derivar la categoría de otra columna confiable (p. ej.
el mes a partir de la fecha), hazlo en una celda de ingeniería de variables y simplemente no incluyas
esa columna en `columnas_texto`.

In [ ]:
class LimpiadorTexto(BaseEstimator, TransformerMixin):
    """Normaliza texto en columnas categóricas: quita espacios y unifica mayúsculas/minúsculas."""
    def __init__(self, columnas=None):
        self.columnas = columnas or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for columna in self.columnas:
            if columna in X.columns:
                X[columna] = X[columna].astype(str).str.strip().str.lower()
        return X

# TODO: lista las columnas categóricas que quieres normalizar por texto (puede quedar vacía).
COLUMNAS_TEXTO_A_NORMALIZAR = []

### 3.4 Selección de variables predictoras y partición entrenamiento/prueba

La partición se hace **una sola vez**, con la semilla y proporción fijadas por el enunciado, y antes
de ajustar cualquier imputador o escalador (para que no se calculen con información del conjunto de
prueba).

In [ ]:
FEATURE_COLS = [c for c in etiquetado.columns if c not in [TARGET] + ID_COLS]
X = etiquetado[FEATURE_COLS]
y = etiquetado[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

columnas_numericas    = X_train.select_dtypes(include='number').columns.tolist()
columnas_categoricas  = [c for c in X_train.columns if c not in columnas_numericas]

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Predictoras numéricas ({len(columnas_numericas)}): {columnas_numericas}')
print(f'Predictoras categóricas ({len(columnas_categoricas)}): {columnas_categoricas}')

### 3.5 `ColumnTransformer`: imputación y estandarización

Los pasos que sí se **ajustan** a los datos (imputación por mediana/moda, `StandardScaler`,
`OneHotEncoder`) van dentro del `ColumnTransformer`, de modo que `Pipeline.fit` los ajuste solo con
`X_train` y `Pipeline.transform`/`predict` los reutilice igual sobre `X_test` y sobre el conjunto de
entrega. Se usa `OneHotEncoder(drop='first', ...)` para evitar la trampa de la variable ficticia
(colinealidad perfecta entre categorías dummy), lo cual también hace más limpio el cálculo de VIF más
adelante.

In [ ]:
def construir_preprocesador(columnas_numericas, columnas_categoricas):
    pipeline_numerico = Pipeline([
        ('imputar', SimpleImputer(strategy='median')),
        ('escalar', StandardScaler()),
    ])
    pipeline_categorico = Pipeline([
        ('imputar', SimpleImputer(strategy='most_frequent')),
        ('codificar', OneHotEncoder(drop='first', handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numericas', pipeline_numerico, columnas_numericas),
        ('categoricas', pipeline_categorico, columnas_categoricas),
    ], remainder='drop')


def construir_pipeline_modelo(modelo, columnas_numericas, columnas_categoricas,
                               reglas_dominio=None, columnas_texto=None):
    """Ensambla el pipeline completo: corrección de dominio -> normalización de texto ->
    imputación/escalamiento/codificación -> modelo."""
    pasos = []
    if reglas_dominio:
        pasos.append(('dominio', CorrectorDominio(reglas_dominio)))
    if columnas_texto:
        pasos.append(('texto', LimpiadorTexto(columnas_texto)))
    pasos.append(('columnas', construir_preprocesador(columnas_numericas, columnas_categoricas)))
    pasos.append(('modelo', modelo))
    return Pipeline(pasos)

print('Funciones de construcción del pipeline listas.')

<a name="s4"></a>
## 4. Construcción de los modelos

Se comparan tres variantes de regresión lineal, todas con la misma preparación de datos:

* **Lineal (OLS)** — línea base, sin regularización.
* **RidgeCV** — regularización L2; reduce la magnitud de los coeficientes y ayuda con la
  multicolinealidad (ver sección 6.4).
* **LassoCV** — regularización L1; además puede llevar coeficientes exactamente a cero, funcionando
  como selección automática de variables.

`RidgeCV`/`LassoCV` eligen internamente el mejor `alpha` por validación cruzada, así que no hace falta
una búsqueda externa con `GridSearchCV` para este laboratorio (eso se profundiza en el Laboratorio 2).

In [ ]:
alphas = np.logspace(-3, 3, 50)
cv_interno = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

modelo_lineal = construir_pipeline_modelo(
    LinearRegression(),
    columnas_numericas, columnas_categoricas, REGLAS_DOMINIO, COLUMNAS_TEXTO_A_NORMALIZAR,
)

modelo_ridge = construir_pipeline_modelo(
    RidgeCV(alphas=alphas, cv=cv_interno),
    columnas_numericas, columnas_categoricas, REGLAS_DOMINIO, COLUMNAS_TEXTO_A_NORMALIZAR,
)

modelo_lasso = construir_pipeline_modelo(
    LassoCV(alphas=alphas, cv=cv_interno, max_iter=20000, random_state=RANDOM_STATE),
    columnas_numericas, columnas_categoricas, REGLAS_DOMINIO, COLUMNAS_TEXTO_A_NORMALIZAR,
)

MODELOS = {'Lineal (OLS)': modelo_lineal, 'RidgeCV': modelo_ridge, 'LassoCV': modelo_lasso}
print('Modelos definidos:', list(MODELOS.keys()))

In [ ]:
# --- Validación cruzada sobre TODO el conjunto etiquetado (más robusto que un único split) ---
cv_externo = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def validar_cruzado(pipeline, X, y, nombre, cv):
    resultados = cross_validate(
        pipeline, X, y, cv=cv,
        scoring={'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
    )
    return {
        'modelo': nombre,
        'rmse_cv_media': -resultados['test_rmse'].mean(), 'rmse_cv_std': resultados['test_rmse'].std(),
        'mae_cv_media' : -resultados['test_mae'].mean(),  'mae_cv_std' : resultados['test_mae'].std(),
        'r2_cv_media'  : resultados['test_r2'].mean(),    'r2_cv_std'  : resultados['test_r2'].std(),
    }

resumen_cv = pd.DataFrame([
    validar_cruzado(pipeline, X, y, nombre, cv_externo) for nombre, pipeline in MODELOS.items()
]).round(4)
resumen_cv

> ✍️ **Interpretación:** compara `rmse_cv_media` y su `rmse_cv_std` entre los tres modelos. Un
> modelo con media similar pero menor desviación estándar es **más estable** entre folds — eso importa
> tanto como el promedio a la hora de elegir el modelo final (sección 8).

<a name="s5"></a>
## 5. Evaluación cuantitativa: tabla comparativa

Se reportan MSE, RMSE, MAE y R² **en entrenamiento y en prueba** para cada modelo. La brecha entre
ambos conjuntos es la señal directa de sobreajuste: un modelo que es mucho mejor en entrenamiento que
en prueba memorizó ruido en vez de generalizar.

In [ ]:
def metricas(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': mean_absolute_error(y_true, y_pred),
            'R2': r2_score(y_true, y_pred)}

def evaluar_modelo(pipeline, X_train, y_train, X_test, y_test, nombre):
    pipeline.fit(X_train, y_train)
    filas = [
        {'modelo': nombre, 'conjunto': 'train', **metricas(y_train, pipeline.predict(X_train))},
        {'modelo': nombre, 'conjunto': 'test',  **metricas(y_test,  pipeline.predict(X_test))},
    ]
    return pd.DataFrame(filas)

tabla_comparativa = pd.concat(
    [evaluar_modelo(pipeline, X_train, y_train, X_test, y_test, nombre)
     for nombre, pipeline in MODELOS.items()],
    ignore_index=True,
).round(4)
tabla_comparativa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
tabla_pivot = tabla_comparativa.pivot(index='modelo', columns='conjunto', values='RMSE')
tabla_pivot.plot.bar(ax=ax, color=['#4c72b0', '#c44e52'])
ax.set_ylabel('RMSE (°C)')
ax.set_title('RMSE en entrenamiento vs. prueba, por modelo')
plt.xticks(rotation=0)
plt.show()

> ✍️ **Tu análisis:** ¿algún modelo muestra una brecha grande entre train y test (indicio de
> sobreajuste)? ¿la regularización (Ridge/Lasso) reduce esa brecha frente al modelo lineal simple?
> ¿el mejor RMSE de test coincide con el mejor promedio de validación cruzada de la sección 4? Si no
> coincide, ¿a qué se puede deber (tamaño de la partición de prueba, aleatoriedad del split, etc.)?

<a name="s6"></a>
## 6. Verificación de los supuestos de la regresión lineal

Se verifican los cuatro supuestos clásicos de la regresión lineal usando el **modelo seleccionado**
(ajústalo abajo según lo que concluyas de la sección 5) y sus residuos sobre el conjunto de prueba.

In [ ]:
# TODO: ajusta cuál modelo usar para el análisis de supuestos, según tu conclusión de la sección 5.
MODELO_SUPUESTOS = clone(modelo_ridge)
MODELO_SUPUESTOS.fit(X_train, y_train)

predichos_test = MODELO_SUPUESTOS.predict(X_test)
residuos_test  = y_test.values - predichos_test

print(f'Media de los residuos: {residuos_test.mean():.4f}  (debería ser ≈ 0)')

### 6.1 Linealidad y media cero de los residuos

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(predichos_test, residuos_test, alpha=0.4, s=15, color='#4c72b0')
ax[0].axhline(0, color='#c44e52', lw=1.5)
ax[0].set_xlabel('Valor predicho'); ax[0].set_ylabel('Residuo')
ax[0].set_title('Residuos vs. predichos')

ax[1].scatter(y_test, predichos_test, alpha=0.4, s=15, color='#4c72b0')
lims = [min(y_test.min(), predichos_test.min()), max(y_test.max(), predichos_test.max())]
ax[1].plot(lims, lims, color='#c44e52', lw=1.5)
ax[1].set_xlabel('Valor observado'); ax[1].set_ylabel('Valor predicho')
ax[1].set_title('Observado vs. predicho')
plt.show()

> ✍️ **Interpretación:** si los residuos no muestran un patrón sistemático (curva, embudo) alrededor
> de cero, el supuesto de linealidad es razonable.

### 6.2 Homocedasticidad — test de Breusch-Pagan

$H_0$: la varianza de los residuos es constante (homocedasticidad). Un p-valor pequeño (< 0.05)
lleva a rechazar $H_0$, es decir, evidencia de heterocedasticidad.

In [ ]:
diseno_test = MODELO_SUPUESTOS[:-1].transform(X_test)   # todo el pipeline menos el modelo
exog = sm.add_constant(diseno_test)
bp_stat, bp_pvalue, f_stat, f_pvalue = het_breuschpagan(residuos_test, exog)

print(f'Estadístico LM de Breusch-Pagan : {bp_stat:.3f}')
print(f'p-valor                         : {bp_pvalue:.4f}')
print('Conclusión:', 'se RECHAZA H0 -> heterocedasticidad' if bp_pvalue < 0.05
      else 'NO se rechaza H0 -> homocedasticidad razonable')

### 6.3 Independencia de los residuos — Durbin-Watson

El estadístico va de 0 a 4; valores cercanos a **2** indican ausencia de autocorrelación. Valores
bajos (< 1.5) sugieren autocorrelación positiva; valores altos (> 2.5), autocorrelación negativa.
Como los datos son una serie temporal diaria, conviene calcularlo **respetando el orden cronológico**
(por eso se recomienda ordenar por fecha antes de este cálculo si el índice no lo preserva).

In [ ]:
dw = durbin_watson(residuos_test)
print(f'Estadístico de Durbin-Watson: {dw:.3f}')

### 6.4 Normalidad de los residuos

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(residuos_test, bins=40, color='#4c72b0', edgecolor='white')
ax[0].set_title('Histograma de residuos')

stats.probplot(residuos_test, dist='norm', plot=ax[1])
ax[1].set_title('Q-Q plot de residuos')
plt.show()

jb_stat, jb_pvalue = stats.jarque_bera(residuos_test)
print(f'Jarque-Bera: estadístico={jb_stat:.3f}, p-valor={jb_pvalue:.4f}')
print('Conclusión:', 'se RECHAZA normalidad' if jb_pvalue < 0.05 else 'NO se rechaza normalidad')

### 6.5 Multicolinealidad — Factor de Inflación de la Varianza (VIF)

Se calcula sobre las variables numéricas predictoras (ya imputadas). Como referencia orientativa:
VIF > 5 sugiere colinealidad moderada, VIF > 10 sugiere colinealidad alta.

In [ ]:
X_num_train_imputado = pd.DataFrame(
    SimpleImputer(strategy='median').fit_transform(X_train[columnas_numericas]),
    columns=columnas_numericas,
)
X_vif = sm.add_constant(X_num_train_imputado)

vif = pd.DataFrame({
    'variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif = vif[vif['variable'] != 'const'].sort_values('VIF', ascending=False).reset_index(drop=True)
vif

> ✍️ **Síntesis de los 4 supuestos:** resume en un par de frases cuáles se cumplen y cuáles no.
> Si hay heterocedasticidad y/o multicolinealidad relevante (VIF alto en varias variables), explica por
> qué eso te lleva a **interpretar con cautela la significancia individual de cada coeficiente** (no
> así su capacidad predictiva conjunta), y por qué la regularización (Ridge/Lasso) es una respuesta
> razonable a la multicolinealidad.

<a name="s7"></a>
## 7. Importancia de variables

Se combina una mirada **cuantitativa** (coeficientes del modelo, importancia por permutación) con una
**cualitativa** (interpretación física de las variables más influyentes).

### 7.1 Coeficientes del modelo

Como las variables numéricas están estandarizadas dentro del pipeline, los coeficientes son
comparables entre sí en magnitud (a diferencia de coeficientes sobre variables en su escala original).

In [ ]:
nombres_transformados = MODELO_SUPUESTOS.named_steps['columnas'].get_feature_names_out()
coeficientes = pd.DataFrame({
    'variable': nombres_transformados,
    'coeficiente': MODELO_SUPUESTOS.named_steps['modelo'].coef_,
}).sort_values('coeficiente', key=np.abs, ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
top = coeficientes.head(15).iloc[::-1]
ax.barh(top['variable'], top['coeficiente'], color=np.where(top['coeficiente'] > 0, '#c44e52', '#4c72b0'))
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Top 15 coeficientes por magnitud absoluta')
plt.show()

coeficientes.head(15)

### 7.2 Importancia por permutación

A diferencia de los coeficientes, la importancia por permutación mide directamente el impacto en el
desempeño del modelo (sobre el conjunto de prueba) al romper aleatoriamente la relación de cada
variable con el objetivo — funciona igual para cualquier tipo de modelo.

In [ ]:
resultado_permutacion = permutation_importance(
    MODELO_SUPUESTOS, X_test, y_test,
    n_repeats=30, random_state=RANDOM_STATE, scoring='neg_root_mean_squared_error',
)
importancia_permutacion = pd.DataFrame({
    'variable': X_test.columns,
    'importancia_media': resultado_permutacion.importances_mean,
    'importancia_std': resultado_permutacion.importances_std,
}).sort_values('importancia_media', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
top = importancia_permutacion.head(15).iloc[::-1]
ax.barh(top['variable'], top['importancia_media'], xerr=top['importancia_std'], color='#55a868')
ax.set_title('Top 15 variables por importancia de permutación (aumento de RMSE)')
plt.show()

importancia_permutacion.head(15)

> ✍️ **Tu análisis:**
> * ¿Coinciden las variables más importantes según coeficientes y según permutación? Si no coinciden
>   del todo, ¿por qué podría pasar (p. ej. variables correlacionadas entre sí)?
> * Elige las 3–4 variables más influyentes y da una **interpretación física** de cada una en el
>   contexto de predecir la temperatura máxima de mañana en la estación de Jena.
> * Si usaste LassoCV, ¿qué variables llevó exactamente a cero? ¿qué implica eso en términos de
>   selección automática de características e interpretabilidad del modelo?

In [ ]:
# Si el modelo elegido es Lasso, esta celda lista qué coeficientes quedaron en cero.
if isinstance(MODELO_SUPUESTOS.named_steps['modelo'], type(modelo_lasso.named_steps['modelo'])):
    en_cero = coeficientes[coeficientes['coeficiente'] == 0]
    print(f'Variables llevadas a cero por Lasso: {len(en_cero)} de {len(coeficientes)}')
    display(en_cero)
else:
    print('El modelo seleccionado para el análisis de supuestos no es Lasso; '
          'vuelve a ejecutar esta celda con MODELO_SUPUESTOS = clone(modelo_lasso) si quieres verlo.')

<a name="s8"></a>
## 8. Selección del modelo final y predicciones sobre el conjunto de entrega

Con base en la tabla comparativa (sección 5), la validación cruzada (sección 4) y el análisis de
supuestos/importancia (secciones 6–7), se elige el modelo final, se reentrena con **todo** el conjunto
etiquetado limpio (para aprovechar toda la información disponible) y se predice sobre el conjunto de
entrega.

In [ ]:
# TODO: ajusta según cuál modelo hayas justificado como el mejor.
MODELO_FINAL = clone(modelo_ridge)
MODELO_FINAL.fit(X, y)

print(f'Filas usadas para reentrenar: {len(X):,}')

In [ ]:
entrega = limpieza_estructural(entrega_raw, es_entrenamiento=False)
predicciones = MODELO_FINAL.predict(entrega[FEATURE_COLS])

print(f'Predicciones generadas: {len(predicciones)}')
print(f'Rango  : {predicciones.min():.2f} °C a {predicciones.max():.2f} °C')
print(f'Media  : {predicciones.mean():.2f} °C  (media del conjunto etiquetado: {y.mean():.2f} °C)')

> ✍️ **Verificación de sensatez:** antes de exportar, revisa que el rango y la media de las
> predicciones sean físicamente plausibles para la ubicación y la época del año del conjunto de
> entrega. Si tienes columnas de fecha/mes, vale la pena graficar la predicción promedio por mes contra
> el promedio histórico del conjunto etiquetado, para confirmar que se reproduce el ciclo estacional.

In [ ]:
# --- Exportación del archivo de entrega ---
salida = entrega_raw.copy()
salida[TARGET] = np.round(predicciones, 2)

os.makedirs('entrega', exist_ok=True)
ruta_salida = os.path.join('entrega', 'Datos Test Lab 1.csv')
salida.to_csv(ruta_salida, index=False, encoding='utf-8')

control = pd.read_csv(ruta_salida)
print(f'Archivo escrito : {ruta_salida}')
print(f'Filas           : {len(control)}   (el original tiene {len(entrega_raw)})')
print(f'Columnas        : {control.shape[1]}  (original {entrega_raw.shape[1]} + {TARGET})')
print(f'Valores nulos en la predicción : {control[TARGET].isna().sum()}')

<a name="s9"></a>
## 9. Análisis de resultados

Responde con tus propios resultados (no hay una respuesta "correcta" única — lo que se evalúa es que
la respuesta esté sustentada en lo que obtuviste en las secciones anteriores).

**Análisis cuantitativo**

- ¿Cuál modelo obtuvo el mejor desempeño en el conjunto de prueba? _completa_
- ¿Coincide con el mejor promedio de validación cruzada? Si no, ¿por qué? _completa_
- ¿El modelo con mejor métrica promedio es necesariamente el más adecuado? Justifica considerando
  también la desviación estándar. _completa_
- ¿Qué supuestos de la regresión lineal se cumplen y cuáles no, y qué implicaciones tiene eso sobre
  cómo interpretas el modelo? _completa_

**Análisis cualitativo**

- ¿Qué variables resultaron más relevantes y qué interpretación física tienen? _completa_
- Si usaste Lasso, ¿qué variables fueron llevadas a cero? ¿qué implica para la interpretabilidad?
  _completa_
- ¿Qué decisiones estratégicas podría tomar AlpesPlanck a partir de estos resultados? _completa_

**Reflexión conceptual**

- ¿Qué relación observas entre la calidad de los datos, la complejidad del modelo y la capacidad de
  generalización? _completa_
- ¿Qué posibles fuentes de sesgo identificas en los datos o en el proceso de modelado (por ejemplo,
  cobertura geográfica de una sola estación, período de años cubierto, variables ausentes como la
  temperatura del día actual)? _completa_

<a name="s10"></a>
## 10. Uso de herramientas de IA generativa

*Sección obligatoria según los principios del curso ISIS-2611 publicados en Bloque Neón.*

### 10.1 Declaración del uso

| Herramienta | Tipo de uso |
|:---|:---|
| _completa_ | _completa_ |

### 10.2 Prompts utilizados

1. _completa con el prompt principal utilizado, textual o resumido_

### 10.3 Análisis crítico del resultado

* **¿Qué partes del contenido generado fueron correctas y útiles?** _completa_
* **¿Qué errores, imprecisiones o limitaciones se identificaron?** _completa_
* **¿Qué decisiones técnicas fueron modificadas respecto a la respuesta de la IAG y por qué?** _completa_
* **¿Qué conceptos del curso permitieron evaluar o mejorar la respuesta generada?** _completa_

### 10.4 Aportes propios

* **¿Qué fue desarrollado, modificado o decidido por el estudiante?** _completa_
* **¿Qué ajustes se realizaron sobre el código o la explicación original?** _completa_
* **¿Qué aprendizajes se obtuvieron del proceso?** _completa_

<a name="s11"></a>
## 11. Guion para el video explicativo (máx. 3 minutos)

Plantilla orientativa — ajusta los tiempos a tu propio contenido:

| Tiempo | Contenido |
|:---:|:---|
| 0:00–0:20 | Contexto del caso AlpesPlanck y objetivo del laboratorio |
| 0:20–0:50 | Principales problemas de calidad de datos encontrados y cómo se resolvieron |
| 0:50–1:40 | Modelos construidos, tabla comparativa y evidencia de sobreajuste/generalización |
| 1:40–2:10 | Supuestos de la regresión lineal: cuáles se cumplen y cuáles no, y qué implica |
| 2:10–2:40 | Variables más importantes y su interpretación para AlpesPlanck |
| 2:40–3:00 | Conclusión y recomendación principal para el equipo de ingeniería de AlpesPlanck |

---
## Conclusiones

_Resume aquí, en 4–6 puntos, los hallazgos y decisiones más importantes del laboratorio: el efecto de
la limpieza de datos, el modelo elegido y por qué, qué supuestos se cumplieron, las variables más
influyentes, y la recomendación principal para AlpesPlanck._